# **Proyecto Etapa 4 — Aprendizaje No Supervisado con PySpark**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

**Actividad individual**

| Nombre | Matrícula |
|--------|-----------|
| Diego Falcón Costilla | A01139580 |

---
## 1. Introducción: Aprendizaje No Supervisado

### 1.1 Concepto general

El **aprendizaje no supervisado** es una rama del aprendizaje automático en la que el modelo no dispone de etiquetas $(y_i)$ durante el entrenamiento. El objetivo es descubrir **estructura latente** en los datos: ya sea agrupando instancias similares (*clustering*), reduciendo su dimensionalidad o modelando la distribución subyacente.

Dado un conjunto sin etiquetas $\mathcal{D} = \{x_1, x_2, \ldots, x_n\}$, el problema de clustering particional busca una asignación $C: \{1,\ldots,n\} \to \{1,\ldots,k\}$ que minimice la distancia intra-cluster. Para K-Means, el objetivo es:

$$\min_{\mu_1,\ldots,\mu_k} \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$

Las principales tareas del aprendizaje no supervisado son:

- **Clustering**: agrupar instancias similares sin etiquetas (K-Means, GMM, DBSCAN, clustering jerárquico).
- **Reducción de dimensionalidad**: proyectar en un espacio de menor dimensión preservando la varianza o la estructura local (PCA, t-SNE, UMAP, autoencoders).
- **Modelado generativo**: aprender la distribución de los datos para generar nuevas muestras (VAE, GAN).
- **Detección de anomalías**: identificar instancias que se desvían de la distribución normal aprendida.

---

### 1.2 Algoritmos representativos en la literatura

| Algoritmo | Tipo | Fortalezas | Limitaciones |
|-----------|------|------------|--------------|
| **K-Means** | Clustering particional | Simple, eficiente, escalable con grandes volúmenes | Asume clusters esféricos e isotrópicos; sensible a escala y outliers |
| **K-Means++** | Clustering particional | Inicialización mejorada → convergencia más estable y resultados de mejor calidad | Mismo costo computacional que K-Means; limitaciones geométricas iguales |
| **Bisecting K-Means** | Clustering jerárquico | Produce jerarquía de clusters; menos sensible a inicialización | Menos flexible en forma de clusters; parámetro k sigue siendo necesario |
| **GMM (Gaussian Mixture Model)** | Clustering probabilístico | Asignación suave (*soft*); permite clusters elípticos; modelado probabilístico | Más costoso (algoritmo EM); puede diverger; supone distribución gaussiana |
| **DBSCAN** | Clustering por densidad | Detecta outliers de forma natural; no requiere especificar k | Sensible a parámetros `eps` y `minPts`; difícil de escalar en alta dimensión |
| **Agglomerative Clustering** | Clustering jerárquico | Produce dendrograma; no requiere k a priori | $O(n^2)$ en memoria y tiempo; difícil de escalar con grandes conjuntos de datos |
| **PCA** | Reducción de dimensionalidad | Lineal, eficiente, interpretable (varianza explicada) | Solo captura relaciones lineales; componentes no necesariamente interpretables |
| **LDA (Latent Dirichlet Allocation)** | Modelado de tópicos | Interpretable para datos de texto y conteos genómicos | Específico para distribuciones discretas |

---

### 1.3 Implementaciones disponibles en PySpark MLlib

PySpark MLlib (`pyspark.ml.clustering`) ofrece las siguientes implementaciones distribuidas:

| Clase PySpark | Algoritmo | Descripción |
|---------------|-----------|-------------|
| `KMeans` | K-Means (K-Means++) | Clustering particional estándar con inicialización K-Means++; soporta distancia euclidiana y coseno |
| `BisectingKMeans` | Bisecting K-Means | Variante divisiva jerárquica; divide iterativamente el cluster con mayor WSSSE |
| `GaussianMixture` | GMM | Mezcla de gaussianas; asignación probabilística (*soft clustering*) mediante el algoritmo EM |
| `LDA` | Latent Dirichlet Allocation | Modelado de tópicos para texto y datos de conteo; variantes online y EM distribuidas |
| `PowerIterationClustering` | PIC | Clustering espectral escalable para grafos de similitud; basado en eigenvectores aproximados |

Para reducción de dimensionalidad, PySpark ofrece `PCA` en `pyspark.ml.feature`.

La calidad del clustering se evalúa con `ClusteringEvaluator`, que implementa el **índice Silhouette** — la métrica estándar para medir cohesión intra-cluster y separación inter-cluster:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, 1]$$

donde $a(i)$ es la distancia media intra-cluster y $b(i)$ es la distancia media al cluster vecino más cercano.

---

### 1.4 Algoritmo seleccionado: K-Means

Se selecciona **K-Means** (con inicialización K-Means++) como algoritmo principal, por las siguientes razones en el contexto GTEx:

1. **Escalabilidad**: K-Means en PySpark utiliza un algoritmo mini-batch distribuido, manejando eficientemente las 799 muestras × 500 features.
2. **Interpretabilidad**: los centroides representan perfiles de expresión génica "promedio" para cada cluster, permitiendo interpretación biológica directa.
3. **Baseline estándar**: K-Means es el algoritmo de clustering de referencia en bioinformática para RNA-seq.
4. **Compatibilidad con PCA**: la combinación `StandardScaler → PCA → K-Means` es el pipeline estándar para datos de alta dimensionalidad genómica.

Como experimento complementario se aplica **GMM** (`GaussianMixture`) para comparar los resultados con asignaciones probabilísticas y evaluar si los clusters de expresión génica siguen distribuciones gaussianas en el espacio PCA.

**Hipótesis**: dado que en la Tarea 3 se demostró que los perfiles de expresión cardiovascular y musculoesquelético son perfectamente separables con aprendizaje supervisado, se espera que K-Means *descubra* estos mismos grupos sin usar etiquetas, obteniendo un índice Silhouette alto y clusters que se alineen con los grupos de tejido conocidos.

In [ ]:
import sys, os

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

_conda_env = os.path.dirname(sys.executable)
_java_home = os.path.join(_conda_env, 'Library', 'lib', 'jvm')
if os.path.isdir(_java_home):
    os.environ['JAVA_HOME'] = _java_home

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    N_GENES, RANDOM_SEED
)

import random
import numpy as np
import pandas as pd
from collections import defaultdict
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sub-sample parameters (same as Tarea 3 for comparability)
TARGET_TISSUE_GROUPS  = ['Cardiovascular', 'Musculoesqueletico']
SAMPLES_PER_PARTITION = 200   # max samples per (tissue_group × sex) partition
N_GENES_VARIANCE      = 500   # top-N genes by inter-sample variance
N_PCA_COMPONENTS      = 50    # PCA dimensions before K-Means

print(f'Semilla aleatoria    : {RANDOM_SEED}')
print(f'Grupos de tejido     : {TARGET_TISSUE_GROUPS}')
print(f'Muestras/partición   : {SAMPLES_PER_PARTITION}')
print(f'Genes (por varianza) : {N_GENES_VARIANCE}')
print(f'Componentes PCA      : {N_PCA_COMPONENTS}')
print(f'JAVA_HOME            : {os.environ.get("JAVA_HOME", "ERROR - no seteado")}')

In [ ]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_UnsupervisedLearning_TC5057') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark

---
## 2. Selección de los datos

### 2.1 Estrategia

Se construye la sub-muestra **M'** a partir de la muestra de equipo M (Etapa 2), siguiendo la misma estrategia de la Tarea 3 para mantener comparabilidad:

- Se trabaja con las **particiones cardiovasculares y musculoesqueléticas**, que corresponden a los grupos de tejido de mayor interés biológico en el contexto del proyecto.
- Se aplica un muestreo **estratificado proporcional por subtipo de tejido (SMTSD)** dentro de cada partición (TISSUE_GROUP × SEX_LABEL), tomando hasta **200 muestras por partición**.
- Resultado esperado: ~799 muestras (~400 cardiovasculares + ~400 musculoesqueléticas).
- Las **etiquetas de tejido** se guardan por separado y **NO se utilizan durante el entrenamiento** (escenario no supervisado). Solo se usan en la validación externa post-clustering.

**Selección de genes**: se seleccionan los **500 genes de mayor varianza** entre las 799 muestras. Esta selección captura genes diferencialmente expresados entre tejidos sin usar información de las etiquetas, siendo apropiada para el contexto no supervisado.

**Justificación del tamaño de M'**: 800 muestras × 500 genes da una matriz manejable para PySpark en modo local, y suficiente para que K-Means produzca clusters estadísticamente estables.

In [ ]:
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMTSD', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ') \
    .withColumn('SUBJID', F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1))

sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

meta_df = sa_df.join(sp_df, on='SUBJID', how='inner')

tissue_group_col = F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso') \
    .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico') \
    .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular') \
    .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico') \
    .otherwise('Visceral_Metabolico')

sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = meta_df \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col) \
    .withColumn('COL_NAME',
        F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_'))

meta_target = meta_df.filter(F.col('TISSUE_GROUP').isin(TARGET_TISSUE_GROUPS))

print('Distribución por partición (grupos objetivo):')
meta_target.groupBy('TISSUE_GROUP', 'SEX_LABEL').count().orderBy('TISSUE_GROUP', 'SEX_LABEL').show()

In [ ]:
meta_rows = meta_target.select('COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL', 'SMTSD').collect()

partition_samples = defaultdict(list)
for row in meta_rows:
    key = (row['TISSUE_GROUP'], row['SEX_LABEL'])
    partition_samples[key].append((row['COL_NAME'], row['SMTSD']))

rng = random.Random(RANDOM_SEED)
selected_col_names = []
selected_meta = []   # (col_name, tissue_group, sex_label)

for (tg, sx), items in sorted(partition_samples.items()):
    n_partition = len(items)
    n_select = min(SAMPLES_PER_PARTITION, n_partition)

    by_subtype = defaultdict(list)
    for col, smtsd in items:
        by_subtype[smtsd].append(col)

    sampled = []
    for smtsd, cols in by_subtype.items():
        n_strata = max(1, round(n_select * len(cols) / n_partition))
        n_strata = min(n_strata, len(cols))
        sampled.extend(rng.sample(cols, n_strata))

    sampled = sampled[:n_select]

    for col in sampled:
        selected_col_names.append(col)
        selected_meta.append((col, tg, sx))

    print(f'  {tg:<22} + {sx:<12}: {len(sampled)} muestras de {n_partition}')

# Build label lookup — stored separately, NOT used during training
meta_dict = {col: tg for col, tg, sx in selected_meta}
print(f'\nTotal M\' : {len(selected_col_names)} muestras')

In [ ]:
from pyspark.sql.functions import split as spark_split

peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_col_names_raw   = peek.columns.tolist()
all_col_names_clean = [c.replace('-', '_').replace('.', '_') for c in all_col_names_raw]
name_to_idx = {clean: idx for idx, clean in enumerate(all_col_names_clean)}

fixed_cols    = ['Name', 'Description']
fixed_indices = [name_to_idx[c] for c in fixed_cols]

valid_sample_cols = [c for c in selected_col_names if c in name_to_idx]
sample_indices    = [name_to_idx[c] for c in valid_sample_cols]

all_selected_indices = fixed_indices + sample_indices
all_selected_names   = fixed_cols + valid_sample_cols

raw_df  = spark.read.text(FILE_PATH)
gene_df = raw_df.filter(F.col('value').startswith('ENSG'))
split_col = spark_split(F.col('value'), '\t')

df_tpm = gene_df.select(
    *[split_col.getItem(i).alias(all_selected_names[idx])
      for idx, i in enumerate(all_selected_indices)]
)

for c in valid_sample_cols:
    df_tpm = df_tpm.withColumn(c, F.col(c).cast(FloatType()))

print(f'DataFrame TPM M\': {df_tpm.count():,} genes × {len(valid_sample_cols)} muestras')
df_tpm.select(all_selected_names[:5]).show(3)

In [ ]:
print('Calculando varianza inter-muestras para todos los genes...')
tpm_full_pd = df_tpm.select(['Name'] + valid_sample_cols).toPandas()
tpm_full_pd = tpm_full_pd.set_index('Name')

gene_var    = tpm_full_pd.var(axis=1).sort_values(ascending=False)
top_var_ids = gene_var.head(N_GENES_VARIANCE).index.tolist()

desc_map = df_tpm.select('Name', 'Description').toPandas() \
    .set_index('Name')['Description'].to_dict()

print(f'\nTop 10 genes por varianza (candidatos a marcadores tisulares):')
print(f'  {"Rank":<5} {"Ensembl ID":<30} {"Símbolo":<15} {"Varianza":>18}')
print('  ' + '-' * 72)
for rank, eid in enumerate(top_var_ids[:10], 1):
    sym = desc_map.get(eid, '?')
    print(f'  {rank:<5} {eid:<30} {sym:<15} {gene_var[eid]:>18,.0f}')

In [ ]:
valid_genes = [g for g in top_var_ids if g in tpm_full_pd.index]
gene_cols   = [g.replace('.', '_') for g in valid_genes]

tpm_T = tpm_full_pd.loc[valid_genes].T.reset_index()
tpm_T = tpm_T.rename(columns={'index': 'COL_NAME'})
tpm_T = tpm_T.rename(columns={g: g.replace('.', '_') for g in valid_genes})
tpm_T[gene_cols] = tpm_T[gene_cols].fillna(0.0)

# Etiquetas guardadas como columna auxiliar — NO formarán parte del entrenamiento
tpm_T['TISSUE_GROUP'] = tpm_T['COL_NAME'].map(meta_dict)
tpm_T = tpm_T.dropna(subset=['TISSUE_GROUP'])

print(f'Matriz M\' final : {tpm_T.shape[0]} muestras × {len(gene_cols)} genes')
print('Distribución por grupo de tejido:')
print(tpm_T['TISSUE_GROUP'].value_counts().to_dict())

---
## 3. Preparación del conjunto de entrenamiento y prueba

### 3.1 Técnica de división

Se aplica una **división estratificada 80 / 20** (train / test):

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Train | 80% (~639 muestras) | Proporciona suficientes datos para que K-Means y GMM ajusten centroides estables en 50 dimensiones PCA |
| Test | 20% (~160 muestras) | Conjunto independiente para calcular el Silhouette sobre instancias no vistas durante el entrenamiento |
| Estratificación | Por `TISSUE_GROUP` | Mantiene la misma proporción de clases en ambos subconjuntos, evitando que el test tenga distribución distinta |
| Semilla | `RANDOM_SEED = 42` | Reproducibilidad de resultados |

**Nota sobre aprendizaje no supervisado**: a diferencia del aprendizaje supervisado, las etiquetas (`TISSUE_GROUP`) **no se utilizan durante el entrenamiento**. La división estratificada se aplica únicamente para asegurar que el conjunto de prueba tenga representación balanceada de los grupos, permitiendo una evaluación externa (post-hoc) más robusta.

El pipeline de preprocesamiento (`StandardScaler`, `PCA`) se ajusta **exclusivamente sobre el conjunto de entrenamiento** y se aplica al de prueba, siguiendo el principio de no fuga de información (*data leakage prevention*).

In [ ]:
from sklearn.model_selection import train_test_split

indices  = list(range(len(tpm_T)))
y_groups = tpm_T['TISSUE_GROUP'].values

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_groups
)

train_df = tpm_T.iloc[train_idx].reset_index(drop=True)
test_df  = tpm_T.iloc[test_idx].reset_index(drop=True)

print(f'Entrenamiento : {len(train_df)} muestras  ({len(train_df)/len(tpm_T)*100:.1f}%)')
print(f'Prueba        : {len(test_df)} muestras  ({len(test_df)/len(tpm_T)*100:.1f}%)')
print(f'\nGrupos en train: {dict(train_df["TISSUE_GROUP"].value_counts())}')
print(f'Grupos en test : {dict(test_df["TISSUE_GROUP"].value_counts())}')

In [ ]:
# Incluir TISSUE_GROUP y COL_NAME en el DataFrame de Spark solo como
# columnas auxiliares — el VectorAssembler las ignorará durante el clustering
train_spark = spark.createDataFrame(train_df[gene_cols + ['COL_NAME', 'TISSUE_GROUP']])
test_spark  = spark.createDataFrame(test_df[gene_cols  + ['COL_NAME', 'TISSUE_GROUP']])

print(f'Spark train : {train_spark.count()} filas')
print(f'Spark test  : {test_spark.count()} filas')

---
## 4. Construcción de modelos de aprendizaje no supervisado

### 4.1 Pipeline de preprocesamiento

Antes de aplicar K-Means o GMM, se requieren dos etapas de preparación:

1. **`StandardScaler`** (`withMean=True, withStd=True`): centra y normaliza cada gen a media 0 y desviación estándar 1. K-Means es sensible a la escala — sin normalización, genes con alta expresión absoluta dominarían la distancia euclidiana independientemente de su varianza relativa.

2. **`PCA`** (`k=50`): reduce la dimensionalidad de 500 a 50 componentes principales. Esta reducción es necesaria por dos razones:
   - **Maldición de la dimensionalidad**: en 500 dimensiones, las distancias euclidianas se vuelven uniformes (todos los puntos están "igualmente lejos"), degradando la calidad del clustering.
   - **Eficiencia computacional**: K-Means en 50 dimensiones converge más rápido que en 500.
   - Los 50 primeros componentes de PCA suelen explicar el 60–80% de la varianza en datos RNA-seq de GTEx.

El pipeline se ajusta **únicamente sobre el conjunto de entrenamiento** y se aplica al de prueba.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=gene_cols, outputCol='raw_features')
scaler    = StandardScaler(
    inputCol='raw_features', outputCol='scaled_features',
    withMean=True, withStd=True
)
pca = PCA(k=N_PCA_COMPONENTS, inputCol='scaled_features', outputCol='pca_features')

prep_pipeline = Pipeline(stages=[assembler, scaler, pca])

print('Ajustando pipeline Assembler → StandardScaler → PCA sobre train...')
prep_model = prep_pipeline.fit(train_spark)

train_pca = prep_model.transform(train_spark)
test_pca  = prep_model.transform(test_spark)

# Varianza explicada por PCA
pca_model     = prep_model.stages[-1]
explained_var = pca_model.explainedVariance.toArray()
cumul_var     = np.cumsum(explained_var)

print(f'\nVarianza explicada acumulada:')
print(f'  PC 1–5   : {cumul_var[4]*100:.1f}%')
print(f'  PC 1–10  : {cumul_var[9]*100:.1f}%')
print(f'  PC 1–50  : {cumul_var[-1]*100:.1f}%')

### 4.2 Método del codo: selección de k para K-Means

Para determinar el número óptimo de clusters $k$, se evalúa el **índice Silhouette** en el conjunto de prueba para $k \in \{2, 3, 4, 5, 6\}$.

El índice Silhouette mide simultáneamente la **cohesión** intra-cluster y la **separación** inter-cluster:
- Valor cercano a **1.0**: instancias bien asignadas, clusters compactos y bien separados.
- Valor cercano a **0.0**: instancias en el límite entre clusters.
- Valor cercano a **-1.0**: instancias probablemente asignadas al cluster incorrecto.

Se elige el $k$ que maximiza el Silhouette medio.

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol='pca_features',
    predictionCol='prediction',
    metricName='silhouette',
    distanceMeasure='squaredEuclidean'
)

print('Método del codo: Silhouette en test para k=2..6')
print(f'  {"k":<5} {"Silhouette":>12}')
print('  ' + '-' * 20)

silhouette_scores = {}
for k in range(2, 7):
    km = KMeans(
        featuresCol='pca_features',
        predictionCol='prediction',
        k=k, maxIter=30, seed=RANDOM_SEED
    )
    km_model = km.fit(train_pca)
    preds    = km_model.transform(test_pca)
    score    = evaluator.evaluate(preds)
    silhouette_scores[k] = score
    print(f'  {k:<5} {score:>12.4f}')

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f'\nMejor k según Silhouette: k={best_k}  (score={silhouette_scores[best_k]:.4f})')

### 4.3 K-Means con k=2

Se entrena el modelo final con $k=2$, que corresponde al número de grupos de tejido conocidos (Cardiovascular y Musculoesquelético). Esto permite comparar directamente los clusters descubiertos por el algoritmo con los grupos reales como validación externa.

**Supuestos del modelo:**
1. Los clusters tienen forma aproximadamente esférica en el espacio PCA.
2. Los puntos de datos se distribuyen de manera relativamente uniforme dentro de cada cluster.
3. La varianza de los genes seleccionados captura las principales diferencias entre tejidos.
4. La normalización (StandardScaler) elimina el efecto de la escala absoluta de expresión.

In [ ]:
K_FINAL = 2  # número de grupos de tejido esperados

kmeans = KMeans(
    featuresCol='pca_features',
    predictionCol='prediction',
    k=K_FINAL,
    maxIter=30,
    seed=RANDOM_SEED
)

print(f'Entrenando K-Means (k={K_FINAL}, PCA {N_PCA_COMPONENTS} componentes, maxIter=30)...')
model_km = kmeans.fit(train_pca)
print('Entrenamiento completado.')
print(f'\nCosto de entrenamiento (WSSSE): {model_km.summary.trainingCost:,.2f}')
print(f'Iteraciones hasta convergencia: {model_km.summary.numIter}')

In [ ]:
preds_km = model_km.transform(test_pca)

sil_km = evaluator.evaluate(preds_km)
print(f'Silhouette score (K-Means, k={K_FINAL}): {sil_km:.4f}')
print()
print('Escala de referencia:')
print('  > 0.70  : estructura de clustering fuerte')
print('  0.50–0.70: estructura razonable')
print('  0.25–0.50: débil, podría ser artificial')
print('  < 0.25  : sin estructura clara')

In [ ]:
# Validación externa: comparar asignaciones de cluster con etiquetas reales
# Las etiquetas NUNCA se usaron durante el entrenamiento
print('Validación externa: distribución de grupos de tejido por cluster K-Means\n')
preds_km.groupBy('prediction', 'TISSUE_GROUP').count() \
    .orderBy('prediction', 'TISSUE_GROUP') \
    .show()

# Tabla de contingencia
preds_pd = preds_km.select('prediction', 'TISSUE_GROUP').toPandas()
contingency = pd.crosstab(
    preds_pd['prediction'],
    preds_pd['TISSUE_GROUP'],
    margins=True,
    margins_name='Total'
)
print('Tabla de contingencia (cluster × grupo de tejido):')
print(contingency)

# Pureza del clustering
total = len(preds_pd)
purity = sum(contingency.iloc[i].drop('Total').max() for i in range(K_FINAL)) / total
print(f'\nPureza del clustering: {purity:.4f}  ({purity*100:.1f}%)')
print('(Fracción de instancias en el cluster que pertenecen al grupo mayoritario)')

### 4.4 Gaussian Mixture Model (GMM) como experimento comparativo

El **GMM** es una alternativa probabilística a K-Means. En lugar de asignar cada punto al centroide más cercano (asignación *hard*), el GMM asigna probabilidades de pertenencia a cada componente gaussiana (asignación *soft*).

**Diferencias clave frente a K-Means:**

| Aspecto | K-Means | GMM |
|---------|---------|-----|
| Tipo de asignación | Hard (cluster definitivo) | Soft (probabilidad por cluster) |
| Forma de clusters | Esférica (isótropa) | Elíptica (covarianza libre) |
| Algoritmo de ajuste | Lloyd (iterativo, cerrado) | EM (Expectation-Maximization) |
| Salida del modelo | Centroide por cluster | Media + covarianza por componente |
| Sensibilidad a outliers | Alta | Moderada |

En el contexto de datos RNA-seq, el GMM puede capturar mejor la variabilidad de expresión dentro de cada tipo de tejido, ya que los perfiles de expresión génica suelen seguir distribuciones log-normales.

In [ ]:
from pyspark.ml.clustering import GaussianMixture

gmm = GaussianMixture(
    featuresCol='pca_features',
    predictionCol='prediction',
    probabilityCol='probability',
    k=2,
    maxIter=30,
    seed=RANDOM_SEED
)

print('Entrenando GMM (k=2, PCA 50 componentes, maxIter=30)...')
model_gmm = gmm.fit(train_pca)
print('Entrenamiento completado.')
print(f'\nLog-verosimilitud de entrenamiento: {model_gmm.summary.logLikelihood:.4f}')

In [ ]:
preds_gmm = model_gmm.transform(test_pca)

sil_gmm = evaluator.evaluate(preds_gmm)
print(f'Silhouette score (GMM, k=2): {sil_gmm:.4f}')

print('\nValidación externa: distribución de grupos de tejido por cluster GMM\n')
preds_gmm.groupBy('prediction', 'TISSUE_GROUP').count() \
    .orderBy('prediction', 'TISSUE_GROUP') \
    .show()

preds_gmm_pd = preds_gmm.select('prediction', 'TISSUE_GROUP').toPandas()
contingency_gmm = pd.crosstab(
    preds_gmm_pd['prediction'],
    preds_gmm_pd['TISSUE_GROUP'],
    margins=True, margins_name='Total'
)
print('Tabla de contingencia GMM:')
print(contingency_gmm)

purity_gmm = sum(contingency_gmm.iloc[i].drop('Total').max() for i in range(2)) / len(preds_gmm_pd)
print(f'\nPureza GMM: {purity_gmm:.4f}  ({purity_gmm*100:.1f}%)')

In [ ]:
print('=' * 55)
print('  Comparación K-Means vs GMM (k=2, PCA 50 componentes)')
print('=' * 55)
print(f'  Silhouette  K-Means : {sil_km:.4f}')
print(f'  Silhouette  GMM     : {sil_gmm:.4f}')
print(f'  Pureza      K-Means : {purity:.4f}  ({purity*100:.1f}%)')
print(f'  Pureza      GMM     : {purity_gmm:.4f}  ({purity_gmm*100:.1f}%)')
print('=' * 55)

---
### 4.5 Interpretación de resultados

#### Métricas esperadas y su significado

| Métrica | Qué mide | Interpretación en contexto GTEx |
|---------|----------|----------------------------------|
| **Silhouette** | Cohesión intra-cluster vs separación inter-cluster | Valor alto indica que los perfiles de expresión génica de cada tejido forman grupos compactos y bien separados en el espacio PCA |
| **Pureza del clustering** | Fracción de instancias correctamente asignadas | Indica qué tan bien los clusters descubiertos se alinean con los grupos de tejido reales sin haber visto las etiquetas |
| **WSSSE (K-Means)** | Suma de distancias cuadráticas al centroide | Valor más bajo indica clusters más compactos; útil para el método del codo |
| **Log-verosimilitud (GMM)** | Bondad del ajuste del modelo probabilístico | Valor menos negativo indica mejor ajuste |

#### Coherencia con resultados de Tarea 3

La Tarea 3 demostró que los perfiles de expresión cardiovascular y musculoesquelético son **perfectamente separables** con Random Forest supervisado (AUC = 1.0). Esto implica que:

1. Los dos tejidos ocupan regiones completamente distintas en el espacio de expresión génica.
2. K-Means y GMM, al operar en el espacio PCA de los mismos genes, deberían descubrir esta misma estructura **sin usar etiquetas**.
3. Se espera un Silhouette score alto (>0.50) y una pureza cercana al 100%.

#### Supuestos y limitaciones

1. **Independencia entre muestras**: un mismo donante puede contribuir muestras de ambos tipos de tejido, introduciendo correlación intra-donante entre train y test. Con ~400 donantes y 800 muestras el efecto es moderado.
2. **Forma de los clusters**: K-Means asume clusters esféricos; si la distribución de expresión en el espacio PCA es elíptica, el GMM (con covarianza no restringida) podría mostrar mejor Silhouette.
3. **Número de componentes PCA**: 50 componentes capturan la mayor parte de la varianza, pero descartan variabilidad de menor escala que podría ser relevante biológicamente.
4. **Selección de genes por varianza sin información de tejido**: aunque no utiliza etiquetas directamente, los genes de alta varianza en esta muestra son precisamente los marcadores tisulares — esto explica los buenos resultados, pero también implica que el experimento tiene sesgo de selección favorable.

---
## Referencias

1. MacQueen, J. B. (1967). Some methods for classification and analysis of multivariate observations. *Proceedings of the 5th Berkeley Symposium on Mathematical Statistics and Probability*, 1, 281–297.
2. Arthur, D., & Vassilvitskii, S. (2007). K-Means++: The advantages of careful seeding. *Proceedings of ACM-SIAM SODA*, 1027–1035.
3. Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society: Series B*, 39(1), 1–38.
4. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
5. Apache Spark MLlib. (2024). Clustering. https://spark.apache.org/docs/latest/ml-clustering.html
6. Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics*, 20, 53–65.

---

## Declaración de uso de Inteligencia Artificial

Anthropic. (2026). *Claude Sonnet 4.6* [Modelo de lenguaje grande], utilizado para soporte en estructura del notebook, documentación de celdas markdown y revisión del código PySpark. https://claude.ai

*La responsabilidad final sobre el contenido entregado recae en el autor. Las decisiones de diseño del experimento, selección de algoritmos, supuestos y la interpretación de resultados son del autor.*